<a href="https://colab.research.google.com/github/514em/assignment2/blob/main/assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.listdir('/content/drive/My Drive/Colab Notebooks/assignment2')

['images',
 'assignment1.ipynb',
 'picture.jpg',
 '3D7BE144-0A71-4F0A-A4CD-C5EA6A22F0C7.jpg']

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
path = '/content/drive/MyDrive/Colab Notebooks/assignment2/'

# Section 1: Image Stitching

In this section we implement keypoint detection, feature matching, homography estimation,
and image stitching using SIFT, BFMatcher, RANSAC, and both pyramid and linear blending.

## Q1.1 – Load Images and Convert to Grayscale

We load the three provided views (`view1.jpeg`, `view2.jpeg`, `view3.jpeg`) from the
`images/Q1/` folder, convert each from BGR to grayscale with `cv2.cvtColor`, and display
both the colour and grayscale versions side-by-side.  
Grayscale conversion is required because subsequent SIFT computation works on single-channel
intensity images.

In [ ]:
# Q1.1 – Load all 3 images and display colour + grayscale versions

q1_path = path + 'images/Q1/'

# Load images in BGR
img1_bgr = cv2.imread(q1_path + 'view1.jpeg')
img2_bgr = cv2.imread(q1_path + 'view2.jpeg')
img3_bgr = cv2.imread(q1_path + 'view3.jpeg')

# Convert BGR -> RGB for matplotlib display
img1_rgb = cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2RGB)
img2_rgb = cv2.cvtColor(img2_bgr, cv2.COLOR_BGR2RGB)
img3_rgb = cv2.cvtColor(img3_bgr, cv2.COLOR_BGR2RGB)

# Convert to grayscale (single-channel, required for SIFT)
img1_gray = cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2GRAY)
img2_gray = cv2.cvtColor(img2_bgr, cv2.COLOR_BGR2GRAY)
img3_gray = cv2.cvtColor(img3_bgr, cv2.COLOR_BGR2GRAY)

images_rgb  = [img1_rgb,  img2_rgb,  img3_rgb]
images_gray = [img1_gray, img2_gray, img3_gray]
labels = ['view1', 'view2', 'view3']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, (rgb, gray, lbl) in enumerate(zip(images_rgb, images_gray, labels)):
    axes[0, col].imshow(rgb)
    axes[0, col].set_title(f'{lbl} – colour')
    axes[0, col].axis('off')

    axes[1, col].imshow(gray, cmap='gray')
    axes[1, col].set_title(f'{lbl} – grayscale')
    axes[1, col].axis('off')

plt.suptitle('Q1.1 – Original colour images (top) and grayscale (bottom)', fontsize=14)
plt.tight_layout()
plt.show()

## Q1.2 – SIFT Keypoints and Descriptors

**Scale-Invariant Feature Transform (SIFT)** detects *keypoints* – distinctive image
locations (corners, blobs) that are stable across scale and rotation – and computes a
128-dimensional **descriptor** for each one.

### (a) Keypoint visualisation
Each keypoint is drawn as a circle whose **radius** encodes the detected scale and whose
**line** inside the circle shows the dominant orientation.  These properties make SIFT
invariant to scale changes and image rotation.

### (b) Descriptor histograms
A SIFT descriptor is a 128-element vector of gradient-orientation histograms sampled in
a 4×4 grid of cells around the keypoint, each cell contributing an 8-bin histogram.
We overlay the full 128-bin histogram for 10 randomly selected descriptors so we can see
how different keypoints capture different local gradient distributions.

In [ ]:
# Q1.2 – SIFT keypoints and descriptors for view1 and view2

# Create SIFT detector
sift = cv2.SIFT_create()

# Detect keypoints and compute descriptors on the grayscale images
kp1, desc1 = sift.detectAndCompute(img1_gray, None)
kp2, desc2 = sift.detectAndCompute(img2_gray, None)

print(f'view1 – {len(kp1)} keypoints detected')
print(f'view2 – {len(kp2)} keypoints detected')

# ── (a) Visualise keypoints with scale (circle radius) and orientation (line) ──
kp1_img = cv2.drawKeypoints(
    img1_rgb, kp1, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)
kp2_img = cv2.drawKeypoints(
    img2_rgb, kp2, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(kp1_img)
axes[0].set_title(f'view1 – SIFT keypoints ({len(kp1)})')
axes[0].axis('off')
axes[1].imshow(kp2_img)
axes[1].set_title(f'view2 – SIFT keypoints ({len(kp2)})')
axes[1].axis('off')
plt.suptitle('Q1.2(a) – Keypoints: circle radius = scale, line = dominant orientation', fontsize=13)
plt.tight_layout()
plt.show()

# ── (b) Overlapping histograms for 10 descriptors ──────────────────────────
np.random.seed(42)
num_show = 10

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, desc, view_label in zip(axes, [desc1, desc2], ['view1', 'view2']):
    indices = np.random.choice(len(desc), num_show, replace=False)
    x = np.arange(128)
    for idx in indices:
        ax.plot(x, desc[idx], alpha=0.6, linewidth=0.9)
    ax.set_title(f'{view_label} – 10 SIFT descriptor histograms (128 dims)')
    ax.set_xlabel('Descriptor dimension')
    ax.set_ylabel('Value')
    ax.legend([f'kp {i}' for i in indices], fontsize=7, loc='upper right')

plt.suptitle('Q1.2(b) – Overlapping SIFT descriptor vectors for 10 keypoints', fontsize=13)
plt.tight_layout()
plt.show()

## Q1.3 – Descriptor Matching with BFMatcher

**Brute-Force Matching** compares every descriptor from view1 against every descriptor
from view2 using the **L2 (Euclidean) distance** – the standard metric for floating-point
SIFT descriptors.  
The matcher returns the single closest neighbour for each query descriptor.  
We then sort all matches by distance and display the **10 best** (lowest-distance) ones with
`cv2.drawMatches()`, which draws connecting lines between the matched keypoint pairs.
Shorter distance ≈ more similar local appearance ≈ more reliable correspondence.

In [ ]:
# Q1.3 – BFMatcher: display the 10 best matches between view1 and view2

# BFMatcher with L2 norm (standard for SIFT float descriptors)
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)

# Match descriptors (one best match per descriptor)
matches = bf.match(desc1, desc2)

# Sort by Euclidean distance – lowest distance = most similar
matches_sorted = sorted(matches, key=lambda m: m.distance)

top10 = matches_sorted[:10]
print(f'Total matches: {len(matches)}')
print('Top-10 match distances:', [round(m.distance, 2) for m in top10])

# Draw the 10 best matches
match_img = cv2.drawMatches(
    img1_rgb, kp1,
    img2_rgb, kp2,
    top10, None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(18, 6))
plt.imshow(match_img)
plt.title('Q1.3 – Top-10 BFMatcher matches between view1 (left) and view2 (right)', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()